In [1]:
import pandas as pd
import geopandas as gpd

# Load joined tree data
trees = gpd.read_file("../data/processed/trees_with_sensors.geojson")

# Load weather and sensor daily data
weather = pd.read_csv("../data/processed/weather_cleaned.csv")
weather['date'] = pd.to_datetime(weather['date'])

sensor_daily = pd.read_csv("../data/processed/sensors_daily.csv")
soil_daily = pd.read_csv("../data/processed/soil_daily.csv")

print(f"Trees: {trees.shape}")
print(f"Weather: {weather.shape}")
print(f"Sensor daily: {sensor_daily.shape}")
print(f"Soil daily: {soil_daily.shape}")

Trees: (82064, 23)
Weather: (4679, 3)
Sensor daily: (6443, 6)
Soil daily: (33115, 5)


In [2]:
# Weather feature engineering
weather = weather.sort_values('date')

# Rolling temperature averages
weather['avg_temp_7d'] = weather['max_temp'].rolling(7).mean()
weather['avg_temp_14d'] = weather['max_temp'].rolling(14).mean()
weather['avg_temp_30d'] = weather['max_temp'].rolling(30).mean()

# Days since last rainfall
weather['had_rain'] = (weather['rainfall_mm'] > 0).astype(int)
rain_groups = (weather['had_rain'] != weather['had_rain'].shift()).cumsum()
weather['days_since_rain'] = weather.groupby(rain_groups).cumcount()
weather.loc[weather['had_rain'] == 1, 'days_since_rain'] = 0

# Heatwave features
weather['above_35'] = (weather['max_temp'] >= 35).astype(int)
weather['heat_degrees'] = (weather['max_temp'] - 35).clip(lower=0)
weather['heat_degree_days_14d'] = weather['heat_degrees'].rolling(14).sum()

# Consecutive hot days
hot_groups = (weather['above_35'] != weather['above_35'].shift()).cumsum()
weather['consec_hot_days'] = weather.groupby(hot_groups).cumcount() + 1
weather.loc[weather['above_35'] == 0, 'consec_hot_days'] = 0
weather['heatwave_flag'] = (weather['consec_hot_days'] >= 3).astype(int)

print("Weather features created:")
print(weather[['date', 'max_temp', 'avg_temp_7d', 'days_since_rain', 'consec_hot_days', 'heatwave_flag']].tail(10))

Weather features created:
           date  max_temp  avg_temp_7d  days_since_rain  consec_hot_days  \
4669 2026-03-15      26.2    24.364286                1                0   
4670 2026-03-16      19.7    22.907143                2                0   
4671 2026-03-17      21.1    21.635714                0                0   
4672 2026-03-18      21.1    21.185714                0                0   
4673 2026-03-19      19.7    21.357143                0                0   
4674 2026-03-20      20.8    21.528571                1                0   
4675 2026-03-21      21.8    21.485714                2                0   
4676 2026-03-22      30.2    22.057143                3                0   
4677 2026-03-23      27.5    23.171429                4                0   
4678 2026-03-24      27.5    24.085714                5                0   

      heatwave_flag  
4669              0  
4670              0  
4671              0  
4672              0  
4673              0  
4674 

In [3]:
# Summarise sensor data per location
sensor_summary = sensor_daily.groupby('sensor_location').agg({
    'avg_temp': 'mean',
    'avg_humidity': 'mean'
}).reset_index()

sensor_summary.columns = ['sensor_location', 'sensor_avg_temp', 'sensor_avg_humidity']

# Merge to trees
trees = trees.merge(sensor_summary, on='sensor_location', how='left')

print(f"Trees after sensor merge: {trees.shape}")
print(f"\nMissing sensor values:")
print(trees[['sensor_avg_temp', 'sensor_avg_humidity']].isnull().sum())

Trees after sensor merge: (82064, 25)

Missing sensor values:
sensor_avg_temp        0
sensor_avg_humidity    0
dtype: int64


In [4]:
#Summarise soil data per site
soil_summary = soil_daily.groupby('site_id').agg({
    'avg_soil_moisture': 'mean'
}).reset_index()

soil_summary.columns = ['site_id', 'avg_soil_moisture']

# Merge to trees
trees = trees.merge(soil_summary, on='site_id', how='left')

print(f"Trees after soil merge: {trees.shape}")
print(f"\nMissing soil values:")
print(trees[['avg_soil_moisture']].isnull().sum())

Trees after soil merge: (82064, 26)

Missing soil values:
avg_soil_moisture    0
dtype: int64


In [5]:
# Get most recent weather features
latest_weather = weather.dropna().tail(1).squeeze()

# Add weather features to all trees (same weather applies city-wide)
trees['max_temp_latest'] = latest_weather['max_temp']
trees['avg_temp_7d'] = latest_weather['avg_temp_7d']
trees['avg_temp_14d'] = latest_weather['avg_temp_14d']
trees['avg_temp_30d'] = latest_weather['avg_temp_30d']
trees['days_since_rain'] = latest_weather['days_since_rain']
trees['heat_degree_days_14d'] = latest_weather['heat_degree_days_14d']
trees['heatwave_flag'] = latest_weather['heatwave_flag']

print(f"Trees with all features: {trees.shape}")
print(f"\nWeather snapshot date: {latest_weather['date']}")
print(f"Max temp: {latest_weather['max_temp']}°C")
print(f"7-day avg: {latest_weather['avg_temp_7d']:.1f}°C")
print(f"Days since rain: {int(latest_weather['days_since_rain'])}")
print(f"Heatwave: {'Yes' if latest_weather['heatwave_flag'] else 'No'}")

Trees with all features: (82064, 33)

Weather snapshot date: 2026-03-24 00:00:00
Max temp: 27.5°C
7-day avg: 24.1°C
Days since rain: 5
Heatwave: No


In [6]:
#Create final feature table
feature_table = trees[[
    # Tree ID
    'com_id',
    
    # Tree characteristics
    'common_name', 'scientific_name', 'genus', 'family',
    'diameter_breast_height', 'year_planted', 'tree_age',
    'age_description',
    
    # Location
    'latitude', 'longitude', 'precinct',
    
    # Microclimate sensor features
    'sensor_avg_temp', 'sensor_avg_humidity',
    
    # Soil features
    'avg_soil_moisture',
    
    # Weather features
    'max_temp_latest', 'avg_temp_7d', 'avg_temp_14d', 'avg_temp_30d',
    'days_since_rain', 'heat_degree_days_14d', 'heatwave_flag',
    
    # Target variable
    'useful_life_expectency_value', 'risk_class'
]].copy()

print(f"Feature table shape: {feature_table.shape}")
print(f"\nMissing values:\n{feature_table.isnull().sum()}")
print(f"\nTarget distribution:\n{feature_table['risk_class'].value_counts()}")

Feature table shape: (82064, 24)

Missing values:
com_id                          0
common_name                     0
scientific_name                 0
genus                           0
family                          0
diameter_breast_height          0
year_planted                    0
tree_age                        0
age_description                 0
latitude                        0
longitude                       0
precinct                        0
sensor_avg_temp                 0
sensor_avg_humidity             0
avg_soil_moisture               0
max_temp_latest                 0
avg_temp_7d                     0
avg_temp_14d                    0
avg_temp_30d                    0
days_since_rain                 0
heat_degree_days_14d            0
heatwave_flag                   0
useful_life_expectency_value    0
risk_class                      0
dtype: int64

Target distribution:
risk_class
LOW       56003
MEDIUM    21115
HIGH       4946
Name: count, dtype: int64


In [7]:
#Save feature table
feature_table.to_csv("../data/processed/feature_table.csv", index=False)
print("Feature table saved!")
print(f"\nFinal summary:")
print(f"  Rows: {feature_table.shape[0]}")
print(f"  Features: {feature_table.shape[1] - 2} (excluding target columns)")
print(f"  Target classes: {feature_table['risk_class'].value_counts().to_dict()}")

Feature table saved!

Final summary:
  Rows: 82064
  Features: 22 (excluding target columns)
  Target classes: {'LOW': 56003, 'MEDIUM': 21115, 'HIGH': 4946}
